# Modulo 6 · Lezione 3 — Probabilità

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Probabilità: misurare l'incertezza |
| 2 | Eventi, spazio campionario, assiomi |
| 3 | Probabilità condizionata |
| 4 | Teorema di Bayes — il cuore dell'IA probabilistica |
| 5 | Distribuzioni discrete: uniforme e binomiale |
| 6 | Distribuzione normale — la più importante in statistica |
| 7 | Altre distribuzioni utili nell'IA |

In [ ]:
import subprocess, sys
for pkg in ['numpy', 'matplotlib', 'scipy']:
    try: __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
print("Librerie pronte")

---
## 1. Cos'è la probabilità?

```
La probabilità misura il grado di incertezza su un evento.

P(A) ∈ [0, 1]
  P = 0  → impossibile
  P = 1  → certezza
  P = 0.5 → altrettanto probabile che accada o no

Perché conta nell'IA?

  ChatGPT non dà la risposta con certezza, genera la parola
  PIÙ PROBABILE data la sequenza precedente.

  Un classificatore di spam non dice "è spam" ma
  P(spam | email) = 0.97.

  Un modello di diffusione (DALL-E, Stable Diffusion) genera
  immagini campionando da una distribuzione di probabilità.

Due interpretazioni:
  Frequentista: P = frequenza nel lungo periodo
  Bayesiana:    P = grado di credenza (più utile nell'IA)
```

---
## 2. Eventi e spazio campionario

In [ ]:
import random

# Simulazione: lanciare un dado 10000 volte
np.random.seed(42)
N = 10000
lanzamientos = np.random.randint(1, 7, size=N)

print("=== Dado a 6 facce ===")
print(f"Spazio campionario: {set(lanzamientos)}")

# Frequenze
for cara in range(1, 7):
    frec = np.sum(lanzamientos == cara) / N
    barra = "█" * int(frec * 100)
    print(f"  P(faccia={cara}) ≈ {frec:.4f}  {barra}")

print(f"\nSomma totale: {sum(np.sum(lanzamientos == c)/N for c in range(1,7)):.4f}  (deve essere 1)")

# Eventi composti
print("\n=== Eventi composti ===")
P_par  = np.sum(lanzamientos % 2 == 0) / N
P_mayor3 = np.sum(lanzamientos > 3) / N
P_par_y_mayor3 = np.sum((lanzamientos % 2 == 0) & (lanzamientos > 3)) / N
P_par_o_mayor3 = np.sum((lanzamientos % 2 == 0) | (lanzamientos > 3)) / N

print(f"P(pari)         = {P_par:.4f}  (teorica: 0.5)")
print(f"P(>3)           = {P_mayor3:.4f}  (teorica: 0.5)")
print(f"P(pari ∩ >3)    = {P_par_y_mayor3:.4f}  (teorica: {2/6:.4f})  → 4,6")
print(f"P(pari ∪ >3)    = {P_par_o_mayor3:.4f}  (teorica: {4/6:.4f})  → 2,4,5,6")
print(f"P(pari) + P(>3) - P(pari∩>3) = {P_par + P_mayor3 - P_par_y_mayor3:.4f}  ← regola dell'addizione")

# Convergenza alla probabilità teorica
fig, ax = plt.subplots(figsize=(8, 3))
ns = np.arange(1, N+1)
frec_acum = np.cumsum(lanzamientos == 6) / ns
ax.plot(ns, frec_acum, lw=1, color='steelblue', label='Frequenza osservata')
ax.axhline(1/6, color='red', ls='--', label=f'P teorica = 1/6 = {1/6:.4f}')
ax.set_xscale('log'); ax.legend()
ax.set_xlabel("Numero di lanci"); ax.set_ylabel("P(faccia=6)")
ax.set_title("Legge dei grandi numeri")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 3. Probabilità condizionata

```
P(A | B) = P(A ∩ B) / P(B)

"Probabilità di A DATO CHE B è accaduto"

Esempio:
  P(faccia > 4 | faccia pari) = ?

  P(>4 ∩ pari) = P({6})         = 1/6
  P(pari)      = P({2,4,6})     = 3/6 = 1/2

  P(>4 | pari) = (1/6) / (1/2) = 1/3

Indipendenza: A e B sono indipendenti se
  P(A | B) = P(A)   →   P(A ∩ B) = P(A) × P(B)
```

In [ ]:
# Esempio reale: rilevamento dello spam
# P(spam) = 0.30 (il 30% delle email è spam)
# P(contiene "offerta" | spam) = 0.85
# P(contiene "offerta" | non spam) = 0.05

P_spam      = 0.30
P_no_spam   = 0.70
P_oferta_dado_spam    = 0.85
P_oferta_dado_nospam  = 0.05

P_oferta = P_oferta_dado_spam * P_spam + P_oferta_dado_nospam * P_no_spam
print(f"P(l'email contiene 'offerta') = {P_oferta:.4f}")

# Tabella di contingenza
N = 10000
n_spam    = int(N * P_spam)
n_nospam  = N - n_spam

tabla = {
    ("spam",     "offerta"):    int(n_spam   * P_oferta_dado_spam),
    ("spam",     "no offerta"): int(n_spam   * (1 - P_oferta_dado_spam)),
    ("non spam", "offerta"):    int(n_nospam * P_oferta_dado_nospam),
    ("non spam", "no offerta"): int(n_nospam * (1 - P_oferta_dado_nospam)),
}

print("\nTabella di contingenza (su 10000 email):")
print(f"{'':15} {'con offerta':>12} {'senza offerta':>13} {'Totale':>10}")
print(f"{'spam':15} {tabla[('spam','offerta')]:>12} {tabla[('spam','no offerta')]:>13} {n_spam:>10}")
print(f"{'non spam':15} {tabla[('non spam','offerta')]:>12} {tabla[('non spam','no offerta')]:>13} {n_nospam:>10}")
total_oferta = tabla[('spam','offerta')] + tabla[('non spam','offerta')]
print(f"{'Totale':15} {total_oferta:>12} {N-total_oferta:>13} {N:>10}")

---
## 4. Teorema di Bayes

```
P(A | B) = P(B | A) × P(A) / P(B)

Nel linguaggio dell'IA:

P(ipotesi | dati) = P(dati | ipotesi) × P(ipotesi) / P(dati)
      ↑ posterior       ↑ verosimiglianza   ↑ prior

Esempio: è spam dato che contiene "offerta"?

  P(spam | offerta) = P(offerta | spam) × P(spam) / P(offerta)
                    = 0.85 × 0.30 / 0.29
                    ≈ 0.879

Interpretazione bayesiana:
  Prior:     P(spam) = 0.30  (credenza prima di vedere l'email)
  Evidenza:  l'email contiene "offerta"
  Posterior: P(spam|offerta) ≈ 0.88  (credenza aggiornata)
```

In [ ]:
def bayes(P_B_dado_A, P_A, P_B):
    return P_B_dado_A * P_A / P_B

P_spam_dado_oferta = bayes(P_oferta_dado_spam, P_spam, P_oferta)
print(f"P(spam | contiene 'offerta') = {P_spam_dado_oferta:.4f} = {P_spam_dado_oferta*100:.1f}%")

# Aggiornamento bayesiano sequenziale
print("\n=== Aggiornamento bayesiano sequenziale ===")
print("Ogni nuova evidenza aggiorna la probabilità:")

# Caratteristiche dell'email e le loro probabilità condizionate
evidencias = [
    ("contiene 'offerta'",     0.85, 0.05),
    ("contiene 'gratis'",      0.70, 0.03),
    ("mittente sconosciuto",   0.90, 0.20),
    ("senza oggetto",          0.40, 0.10),
]

P_spam_actual = P_spam
print(f"  Prior: P(spam) = {P_spam_actual:.4f}")

for desc, P_ev_spam, P_ev_nospam in evidencias:
    P_ev = P_ev_spam * P_spam_actual + P_ev_nospam * (1 - P_spam_actual)
    P_spam_actual = bayes(P_ev_spam, P_spam_actual, P_ev)
    print(f"  + {desc:<35} → P(spam) = {P_spam_actual:.4f}")

print(f"\nDecisione: {'SPAM' if P_spam_actual > 0.5 else 'NON SPAM'} (soglia 0.5)")

---
## 5. Distribuzioni di probabilità

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

# 1. Uniforme discreta
ax = axes[0, 0]
caras = np.arange(1, 7)
ax.bar(caras, [1/6]*6, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_title("Uniforme discreta (dado)")
ax.set_xlabel("Faccia"); ax.set_ylabel("P(X=k)")
ax.set_ylim(0, 0.25)

# 2. Binomiale: n prove, P successo = p
ax = axes[0, 1]
n, p = 20, 0.3
k = np.arange(0, n+1)
pmf = stats.binom.pmf(k, n, p)
ax.bar(k, pmf, color='salmon', alpha=0.7, edgecolor='black')
ax.set_title(f"Binomiale (n={n}, p={p})")
ax.set_xlabel("k (successi)"); ax.set_ylabel("P(X=k)")
media_b = n * p
ax.axvline(media_b, color='red', ls='--', label=f"μ={media_b}")
ax.legend()

# 3. Poisson
ax = axes[0, 2]
lam = 4  # tasso medio
k = np.arange(0, 15)
pmf_p = stats.poisson.pmf(k, lam)
ax.bar(k, pmf_p, color='mediumpurple', alpha=0.7, edgecolor='black')
ax.set_title(f"Poisson (λ={lam})")
ax.set_xlabel("k"); ax.set_ylabel("P(X=k)")
ax.axvline(lam, color='red', ls='--', label=f"λ={lam}")
ax.legend()

# 4. Normale
ax = axes[1, 0]
x = np.linspace(-4, 4, 300)
for mu, sigma, color, ls in [(0, 1, 'blue', '-'), (0, 2, 'red', '--'), (1, 1, 'green', ':')]:
    ax.plot(x, stats.norm.pdf(x, mu, sigma), color=color, ls=ls,
            label=f"μ={mu}, σ={sigma}", lw=2)
ax.set_title("Normale (Gaussiana)")
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
ax.legend(); ax.grid(True, alpha=0.3)

# 5. Esponenziale
ax = axes[1, 1]
x = np.linspace(0, 5, 300)
for rate, color in [(0.5, 'blue'), (1, 'red'), (2, 'green')]:
    ax.plot(x, stats.expon.pdf(x, scale=1/rate), color=color, lw=2, label=f"λ={rate}")
ax.set_title("Esponenziale")
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
ax.legend(); ax.grid(True, alpha=0.3)

# 6. Regola 68-95-99.7 della normale
ax = axes[1, 2]
x = np.linspace(-4, 4, 300)
ax.plot(x, stats.norm.pdf(x), 'black', lw=2)
for sigma, color, alpha_val in [(1, 'blue', 0.5), (2, 'green', 0.3), (3, 'red', 0.2)]:
    mask = np.abs(x) <= sigma
    ax.fill_between(x[mask], stats.norm.pdf(x[mask]), alpha=alpha_val, color=color,
                    label=f"±{sigma}σ: {stats.norm.cdf(sigma)-stats.norm.cdf(-sigma):.1%}")
ax.set_title("Regola 68-95-99.7")
ax.set_xlabel("σ"); ax.set_ylabel("f(x)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle("Distribuzioni di probabilità", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# scipy.stats: il toolkit delle distribuzioni
print("=== scipy.stats: normale standard ===")
dist = stats.norm(loc=0, scale=1)  # μ=0, σ=1

print(f"PDF in x=0:    {dist.pdf(0):.4f}   (densità, non probabilità)")
print(f"CDF in x=1.96: {dist.cdf(1.96):.4f} (P(X ≤ 1.96))")
print(f"P(-1 ≤ X ≤ 1): {dist.cdf(1) - dist.cdf(-1):.4f}  (±1σ ≈ 68%)")
print(f"P(-2 ≤ X ≤ 2): {dist.cdf(2) - dist.cdf(-2):.4f}  (±2σ ≈ 95%)")
print(f"Percentile 95%: {dist.ppf(0.95):.4f}  (valore x tale che P(X≤x)=0.95)")

print("\n=== Campionare dalle distribuzioni ===")
muestras_normal = stats.norm.rvs(loc=170, scale=8, size=1000, random_state=42)
print(f"1000 altezze simulate (μ=170cm, σ=8cm):")
print(f"  Media osservata:      {muestras_normal.mean():.2f}cm")
print(f"  Dev. std. osservata:  {muestras_normal.std():.2f}cm")
print(f"  Min: {muestras_normal.min():.1f}cm, Max: {muestras_normal.max():.1f}cm")

print("\n=== Binomiale: probabilità cumulate ===")
# Se lanciamo 10 monete, qual è la P di ottenere 7 o più teste?
P_7_o_mas = 1 - stats.binom.cdf(6, n=10, p=0.5)
print(f"P(X ≥ 7) con n=10, p=0.5: {P_7_o_mas:.4f}")

# Verificare simulando
resultados = stats.binom.rvs(n=10, p=0.5, size=100000, random_state=42)
print(f"P(X ≥ 7) simulata:         {np.mean(resultados >= 7):.4f}")

---
## Pratica

### Esercizio 1 — Simulazione Montecarlo

**Consegna:** Stima il numero π con il metodo Montecarlo:

```
Idea: inscrivi un cerchio di raggio 1 in un quadrato di lato 2.
Genera punti casuali nel quadrato.
La frazione di punti dentro il cerchio ≈ π/4.

Un punto (x,y) è dentro il cerchio se x² + y² ≤ 1
```

1. Simula con 1000, 10000 e 100000 punti. Come migliora la stima?
2. Visualizza i punti dentro e fuori dal cerchio
3. Traccia l'errore di stima in funzione del numero di punti

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Stime
for N in [1000, 10000, 100000]:
    x = np.random.uniform(-1, 1, N)
    y = np.random.uniform(-1, 1, N)
    dentro = (x**2 + y**2) <= 1
    pi_est = 4 * np.sum(dentro) / N
    print(f"N={N:>7}: π ≈ {pi_est:.6f}  (errore: {abs(pi_est - np.pi):.6f})")

# Visualizzazione
N_vis = 5000
x = np.random.uniform(-1, 1, N_vis)
y = np.random.uniform(-1, 1, N_vis)
dentro = (x**2 + y**2) <= 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter
axes[0].scatter(x[dentro], y[dentro], s=1, c='steelblue', alpha=0.5, label="dentro")
axes[0].scatter(x[~dentro], y[~dentro], s=1, c='salmon', alpha=0.5, label="fuori")
theta = np.linspace(0, 2*np.pi, 300)
axes[0].plot(np.cos(theta), np.sin(theta), 'black', lw=2)
axes[0].set_aspect('equal'); axes[0].set_title(f"Montecarlo: π ≈ {4*dentro.sum()/N_vis:.4f}")
axes[0].legend(markerscale=10)

# Convergenza
ns = np.arange(100, 50001, 100)
x_all = np.random.uniform(-1, 1, 50000)
y_all = np.random.uniform(-1, 1, 50000)
d_all = (x_all**2 + y_all**2) <= 1
pi_acum = [4 * d_all[:n].sum() / n for n in ns]
axes[1].plot(ns, pi_acum, lw=1, color='steelblue', label='Stima')
axes[1].axhline(np.pi, color='red', ls='--', label=f'π reale = {np.pi:.5f}')
axes[1].set_xlabel("N punti"); axes[1].set_ylabel("Stima di π")
axes[1].set_title("Convergenza"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 2 — Filtro Bayesiano

**Consegna:** Implementa un classificatore Bayesiano Naive per rilevare email di phishing.

```
Caratteristiche dell'email:
  P(phishing) = 0.05  (il 5% delle email è phishing)

  P(caratteristica | phishing)  P(caratteristica | legittima)
  'urgente':          0.70              0.03
  'password':         0.60              0.02
  'clicca qui':       0.80              0.10
  'banca':            0.50              0.05
  'conto bloccato':   0.65              0.01
```

Date queste tre email, qual è la probabilità di phishing?
```
Email A: contiene 'urgente', 'clicca qui'
Email B: contiene 'banca', 'password', 'conto bloccato', 'urgente'
Email C: non contiene nessuna delle parole chiave
```

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np

# Probabilità condizionate
caracteristicas = {
    'urgente':        (0.70, 0.03),
    'password':       (0.60, 0.02),
    'clicca qui':     (0.80, 0.10),
    'banca':          (0.50, 0.05),
    'conto bloccato': (0.65, 0.01),
}

def clasificar(palabras_presentes, P_phish_prior=0.05):
    log_P_phish   = np.log(P_phish_prior)
    log_P_legit   = np.log(1 - P_phish_prior)

    for palabra, (p_phish, p_legit) in caracteristicas.items():
        if palabra in palabras_presentes:
            log_P_phish += np.log(p_phish)
            log_P_legit += np.log(p_legit)
        else:
            log_P_phish += np.log(1 - p_phish)
            log_P_legit += np.log(1 - p_legit)

    # Convertire log-probabilità in probabilità
    max_log = max(log_P_phish, log_P_legit)
    p1 = np.exp(log_P_phish - max_log)
    p2 = np.exp(log_P_legit - max_log)
    P_phish_post = p1 / (p1 + p2)
    return P_phish_post

emails = {
    "Email A": {'urgente', 'clicca qui'},
    "Email B": {'banca', 'password', 'conto bloccato', 'urgente'},
    "Email C": set(),
}

for nombre, palabras in emails.items():
    p = clasificar(palabras)
    decision = "⚠️  PHISHING" if p > 0.5 else "✓  Legittima"
    print(f"{nombre}: P(phishing) = {p:.4f}  → {decision}")
    if palabras:
        print(f"  Parole: {palabras}")
```
</details>

---
### Esercizio 3 — Distribuzione normale e percentili

**Consegna:** I punteggi di un test attitudinale seguono una distribuzione normale con μ=100, σ=15.

1. Che percentuale ottiene più di 130 punti?
2. Che percentuale sta tra 85 e 115?
3. Qual è il punteggio minimo per stare nel top 5%?
4. Simula 1000 studenti e visualizza l'istogramma insieme alla PDF teorica
5. Quanti studenti su 1000 supereranno 145 punti (3σ)?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

mu, sigma = 100, 15
dist = stats.norm(loc=mu, scale=sigma)

# 1.
p_mas_130 = 1 - dist.cdf(130)
print(f"P(X > 130) = {p_mas_130:.4f} = {p_mas_130*100:.2f}%")

# 2.
p_85_115 = dist.cdf(115) - dist.cdf(85)
print(f"P(85 ≤ X ≤ 115) = {p_85_115:.4f} = {p_85_115*100:.2f}%")

# 3.
top5 = dist.ppf(0.95)
print(f"Top 5%: punteggio minimo = {top5:.1f}")

# 4. Simulazione + visualizzazione
muestras = dist.rvs(size=1000, random_state=42)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(muestras, bins=30, density=True, alpha=0.6, color='steelblue', label='Simulati (n=1000)')
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
ax.plot(x, dist.pdf(x), 'r-', lw=2, label='PDF teorica')
ax.axvline(top5, color='orange', ls='--', label=f'Top 5% ({top5:.1f})')
ax.axvline(130, color='purple', ls='--', label='130 punti')
ax.set_xlabel('Punteggio'); ax.set_ylabel('Densità')
ax.set_title('Distribuzione dei punteggi (μ=100, σ=15)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# 5.
p_145 = 1 - dist.cdf(145)
esperados = p_145 * 1000
observados = np.sum(muestras > 145)
print(f"\nP(X > 145) = {p_145:.5f}")
print(f"Attesi su 1000: {esperados:.2f}")
print(f"Osservati nella simulazione: {observados}")
```
</details>

---
### Esercizio 4 — Teorema del Limite Centrale

**Consegna:** Il Teorema del Limite Centrale (TLC) stabilisce che la media di molti campioni indipendenti tende a una distribuzione normale, indipendentemente dalla distribuzione originale.

Dimostralo empiricamente:
1. Genera 10000 campioni di dimensione `n` da una distribuzione uniforme su [0,1]
2. Calcola la media di ogni campione
3. Visualizza l'istogramma delle medie per n=1, 5, 30, 100
4. Per quale n la distribuzione delle medie appare già normale?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
N_muestras = 10000
ns = [1, 5, 30, 100]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, n in zip(axes, ns):
    # Generare N_muestras campioni di dimensione n
    datos   = np.random.uniform(0, 1, size=(N_muestras, n))
    medias  = datos.mean(axis=1)

    ax.hist(medias, bins=50, density=True, alpha=0.7, color='steelblue')

    # Curva normale teorica (TLC: μ=0.5, σ=1/(2√3·√n))
    mu_teo    = 0.5
    sigma_teo = (1/12)**0.5 / np.sqrt(n)  # σ_originale/√n, σ_uniforme = 1/√12
    x = np.linspace(medias.min(), medias.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu_teo, sigma_teo), 'r-', lw=2)

    # Test di normalità
    _, p_valor = stats.shapiro(medias[:500])  # Shapiro con sottocampione
    normal = "Normale ✓" if p_valor > 0.05 else f"Non normale (p={p_valor:.3f})"

    ax.set_title(f"n = {n}\n{normal}", fontsize=9)
    ax.set_xlabel("Media campionaria")
    if ax == axes[0]: ax.set_ylabel("Densità")

plt.suptitle("Teorema del Limite Centrale (distribuzione uniforme → media)", fontsize=11)
plt.tight_layout()
plt.show()
print("→ A partire da n≈30, la distribuzione delle medie è già praticamente normale")
```
</details>

---
### Esercizio 5 (sfida) — Generatore di dati sintetici

**Consegna:** Crea un generatore di dataset sintetici per classificazione binaria:

```
Classe 0 (clienti che non comprano):
  - età:      Normale(μ=45, σ=12)
  - reddito:  Normale(μ=28000, σ=8000)
  - visite:   Poisson(λ=2)

Classe 1 (clienti che comprano):
  - età:      Normale(μ=35, σ=10)
  - reddito:  Normale(μ=45000, σ=12000)
  - visite:   Poisson(λ=8)
```

1. Genera 500 clienti per ciascuna classe
2. Visualizza le distribuzioni di ogni caratteristica per classe
3. Quale caratteristica separa meglio le due classi?
4. Implementa un semplice classificatore Bayesiano basato su queste distribuzioni

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
N = 500

# Generare i dati
clase0 = np.column_stack([
    stats.norm.rvs(45, 12, N), stats.norm.rvs(28000, 8000, N), stats.poisson.rvs(2, size=N)
])
clase1 = np.column_stack([
    stats.norm.rvs(35, 10, N), stats.norm.rvs(45000, 12000, N), stats.poisson.rvs(8, size=N)
])

nombres = ["Età", "Reddito", "Visite"]

# Visualizzazione
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (ax, nombre) in enumerate(zip(axes, nombres)):
    ax.hist(clase0[:, i], bins=30, alpha=0.6, density=True, color='salmon', label='Non compra')
    ax.hist(clase1[:, i], bins=30, alpha=0.6, density=True, color='steelblue', label='Compra')
    ax.set_title(nombre); ax.legend()
plt.tight_layout(); plt.show()

# Classificatore Bayesiano
def log_verosimilitud_normal(x, mu, sigma):
    return stats.norm.logpdf(x, mu, sigma)

def clasificar(edad, ingresos, visitas, P_compra=0.5):
    log_p0  = np.log(1 - P_compra)
    log_p0 += log_verosimilitud_normal(edad, 45, 12)
    log_p0 += log_verosimilitud_normal(ingresos, 28000, 8000)
    log_p0 += stats.poisson.logpmf(int(visitas), 2)

    log_p1  = np.log(P_compra)
    log_p1 += log_verosimilitud_normal(edad, 35, 10)
    log_p1 += log_verosimilitud_normal(ingresos, 45000, 12000)
    log_p1 += stats.poisson.logpmf(int(visitas), 8)

    return 1 if log_p1 > log_p0 else 0

# Valutare sui dati di test
X = np.vstack([clase0, clase1])
y = np.array([0]*N + [1]*N)
pred = [clasificar(*fila) for fila in X]
acc = np.mean(np.array(pred) == y)
print(f"Accuracy del classificatore Bayesiano: {acc:.4f} = {acc*100:.1f}%")
```
</details>

---
## Riepilogo

```python
from scipy import stats

# Distribuzioni
dist_normal = stats.norm(loc=0, scale=1)
dist_normal.pdf(0)        # densità
dist_normal.cdf(1.96)     # P(X ≤ 1.96)
dist_normal.ppf(0.95)     # percentile 95
dist_normal.rvs(size=100) # campioni casuali

# Bayes
# P(A|B) = P(B|A) × P(A) / P(B)
P_posterior = P_verosimilitud * P_prior / P_evidencia
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 4** misuriamo i dati che già abbiamo: statistica descrittiva con media, mediana, deviazione standard e correlazione.